# 01 — PF Exercise Generation Experiments

Experimenting with prompting strategies to transform well-structured math problems (GSM8K) into ill-structured Productive Failure exercises guided by Kapur's criteria.

**Pipeline Phase 1:** `SeedProblem → [LLM + prompt] → PFExercise`

In [2]:
import sys
sys.path.insert(0, "..")

import json
import yaml
from pathlib import Path
from src.utils.llm_client import LLMClient

client = LLMClient()
assert client.is_available(), "vLLM not reachable — run ./scripts/euler/connect.sh"
print(f"Connected to {client.model}")

Connected to Qwen/Qwen3-14B


In [4]:
# Load datasets
def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

gsm_train = load_jsonl("../data/gsm8k/train.jsonl")

pf_exercises = []
for f in sorted(Path("../data/pf_exercises").glob("*.yaml")):
    with open(f) as fh:
        data = yaml.safe_load(fh)
    pf_exercises.extend(data["exercises"])

print(f"Loaded {len(gsm_train)} GSM8K seeds, {len(pf_exercises)} PF examples")

Loaded 7473 GSM8K seeds, 16 PF examples


---

## Sample Seed Problems (GSM8K)

These are well-structured problems we want to transform into ill-structured PF exercises.

In [5]:
# Pick a few diverse seed problems from GSM8K
import random
random.seed(42)

# Select problems with varying complexity (by solution steps)
seeds = []
for p in gsm_train:
    steps = p["answer"].count("\n") + 1
    p["_steps"] = steps
    p["_final_answer"] = p["answer"].split("####")[-1].strip()

# Short (2-3 steps), medium (4-5 steps), long (6+ steps)
short = [p for p in gsm_train if p["_steps"] <= 3]
medium = [p for p in gsm_train if 4 <= p["_steps"] <= 5]
long = [p for p in gsm_train if p["_steps"] >= 6]

samples = random.sample(short, 2) + random.sample(medium, 2) + random.sample(long, 1)

for i, p in enumerate(samples):
    print(f"{'='*60}")
    print(f"Seed {i+1} ({p['_steps']} steps):")
    print(f"  {p['question']}")
    print(f"  → Answer: {p['_final_answer']}")
    print()

Seed 1 (3 steps):
  A desert garden’s sprinkler system runs twice a day during the cool morning and evening hours. It waters the garden with four liters of water in the morning and six liters in the evening. How many days does it take the sprinkler system to use 50 liters of water?
  → Answer: 5

Seed 2 (3 steps):
  A movie theater has 6 screens which show movies back-to-back all day. If the movie theater is open for 8 hours, and each movie lasts 2 hours, how many movies are shown in total throughout the day?
  → Answer: 24

Seed 3 (5 steps):
  In 6 months Bella and Bob will be celebrating their 4th anniversary.  How many months ago did they celebrate their 2nd anniversary?
  → Answer: 18

Seed 4 (4 steps):
  Chuck breeds dogs.  He has 3 pregnant dogs.  They each give birth to 4 puppies.  Each puppy needs 2 shots and each shot costs $5.  How much did the shots cost?
  → Answer: 120

Seed 5 (7 steps):
  Charles is jogging while listening to his favorite album. He runs at 6 MPH when he's

---

## What Makes a Good PF Exercise? (Design Principles from Literature)

### Kapur's 5 Criteria of Ill-Structuredness (Kapur & Lee, 2009)

| # | Criterion | Description | What it means for generation |
|---|-----------|-------------|------------------------------|
| 1 | **Parameter complexity** | Multiple variables, non-obvious relationships | Problem has more info than a simple formula can handle |
| 2 | **Solution complexity** | No single correct approach; admits multiple RSMs | Students can try ratios, diagrams, trial-and-error, etc. |
| 3 | **Problem space complexity** | Large search space, many paths to explore | Not solvable in 1-2 steps; requires genuine exploration |
| 4 | **Accessibility** | Uses prior knowledge but is NOT directly solvable with it | Students CAN start (not frustrated) but CANNOT finish (fail productively) |
| 5 | **Collaboration potential** | Supports discussion, explanation, comparison | Problem affords multiple perspectives worth debating |

### Key Design Features (from Kapur & Bielaczyc 2012, Loibl & Rummel 2014, Puech et al. 2025)

**What a PF exercise MUST have:**
- **Narrative context** — real-world scenario for affective engagement
- **Conceptual trap** — intuitive approaches produce plausible but WRONG answers
- **Multiple RSMs** — different students naturally produce DIFFERENT solution attempts
- **Sweet-spot calibration** — challenging enough to fail, accessible enough to try

**What a PF exercise must NOT have:**
- A direct formula application path
- Only one possible approach
- Frustration (too far beyond prior knowledge)
- Trivial solution visible from the problem statement

### The "Seed → PF" Transformation Pattern

```
WELL-STRUCTURED (seed):          ILL-STRUCTURED (PF):
─────────────────────────         ──────────────────────
• Clear formula to apply          • No obvious formula
• Single correct method           • Multiple plausible methods (all fail)
• All info neatly given           • Must make assumptions, interpret context
• Closed answer                   • Open exploration required
• 2-3 step solution               • Rich problem space
```

### PF Score Rubric (from Puech et al. 2025, StratL field study)

A generated PF exercise can be scored on whether students:
1. Give an **intuitive definition** of the target concept
2. Think about **comparing to a baseline** (central value, reference point)
3. Consider **absolute differences** (magnitude matters, not direction)
4. Think about **aggregating** all differences into a single score

### Transformation Strategy

Given a GSM8K seed problem, the LLM should:
1. **Identify the core concept** (what mathematical idea does the answer require?)
2. **Remove the scaffolding** (don't tell them which formula/method to use)
3. **Add narrative context** (make it a story that engages)
4. **Design the conceptual trap** (ensure intuitive methods give wrong answers)
5. **Ensure RSM diversity** (multiple entry points for different students)